# 32 — Speculative RAG

Generate a draft answer first (from LLM training knowledge), extract its factual claims, retrieve evidence only for claims that need checking, then revise just the unsupported parts.

**What you'll learn**
- Why standard RAG always retrieves — and when that's wasteful
- Speculative generation: answer first, fact-check second
- Claim extraction: turning a prose answer into individual checkable facts
- Conditional revision: retrieval only fires when claims are unsupported
- The SUPPORTED / CONTRADICTED / UNRELATED label pattern

**Companion examples:** 17-corrective-rag (correct after retrieval), 27-self-rag (decide per-sentence whether to retrieve)

In [ ]:
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    %pip install -q langchain langchain-openai langchain-chroma python-dotenv langgraph chromadb

In [ ]:
import os

from dotenv import load_dotenv

if not IN_COLAB:
    load_dotenv()
if IN_COLAB and not os.environ.get("OPENAI_API_KEY"):
    from google.colab import userdata

    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
assert os.environ.get("OPENAI_API_KEY"), "Set OPENAI_API_KEY before running"

In [ ]:
# ── 3. Speculative RAG vs standard RAG ────────────────────────────────────────
# Standard RAG:    retrieve → generate  (always retrieves, even for easy questions)
# Speculative RAG: generate → verify → (maybe) revise
#
# The key insight: LLMs already know many facts from training.
# Retrieval is expensive. Why retrieve if the model is already right?
#
# Speculative RAG answers first, then uses retrieval as a TARGETED fact-checker.
# Only claims flagged UNRELATED or CONTRADICTED trigger the revision node.

print("Standard RAG:    retrieve → generate  (always retrieves)")
print("Speculative RAG: draft → extract_claims → retrieve_and_grade → (maybe) revise")
print("\nRetrieval is skipped entirely when all claims are SUPPORTED.")

In [ ]:
# ── 4. Build the speculative RAG graph ────────────────────────────────────────
from typing import TypedDict

from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langgraph.graph import END, StateGraph

DOCS = [
    "LangGraph was created by LangChain Inc. and released in January 2024.",
    "LangGraph is built on top of LangChain and uses its message and tool abstractions.",
    "LangGraph supports human-in-the-loop workflows via interrupt() and Command(resume=...).",
    "LangGraph StateGraph compiles into a runnable that can be invoked or streamed.",
    "LangChain was founded by Harrison Chase in 2022 and raised a Series A in 2023.",
    "MemorySaver is a built-in LangGraph checkpointer that stores state in memory.",
    "LangGraph supports parallel execution using the Send API for map-reduce patterns.",
    "The LangGraph prebuilt create_react_agent handles the ReAct loop automatically.",
    "LangChain's LCEL (LangChain Expression Language) uses the | pipe operator for chains.",
    "LangGraph graphs can have conditional edges that route based on node return values.",
]

DRAFT_SYSTEM = "Answer the question in 3-4 sentences using your training knowledge. Be specific."
CLAIM_SYSTEM = (
    "Extract the distinct factual claims from the given answer as a numbered list. "
    "Each claim should be one checkable fact. Output ONLY the numbered list."
)
SUPPORT_SYSTEM = (
    "Given a claim and retrieved context, respond with exactly one word: "
    "SUPPORTED, CONTRADICTED, or UNRELATED."
)
REVISE_SYSTEM = (
    "Given an original answer and fact-check results, revise ONLY the unsupported "
    "parts. Keep well-supported content unchanged. Return the full revised answer."
)


class SpeculativeState(TypedDict):
    query: str
    draft: str
    claims: list
    evidence: list
    support_labels: list
    revised: str


llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

print("Building vector store...")
vectorstore = Chroma.from_documents([Document(page_content=d) for d in DOCS], embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})
print(f"{len(DOCS)} documents indexed")


def draft(state: SpeculativeState) -> dict:
    result = llm.invoke([SystemMessage(DRAFT_SYSTEM), HumanMessage(state["query"])])
    print(f"  [draft] {result.content[:80]}...")
    return {
        "draft": result.content,
        "claims": [],
        "evidence": [],
        "support_labels": [],
        "revised": "",
    }


def extract_claims(state: SpeculativeState) -> dict:
    result = llm.invoke([SystemMessage(CLAIM_SYSTEM), HumanMessage(f"Answer:\n{state['draft']}")])
    lines = [l.strip() for l in result.content.strip().split("\n") if l.strip()]
    claims = [l.split(". ", 1)[-1] if l[0].isdigit() else l for l in lines]
    print(f"  [extract_claims] {len(claims)} claims")
    return {"claims": claims}


def retrieve_and_grade(state: SpeculativeState) -> dict:
    evidence, labels = [], []
    for claim in state["claims"]:
        docs = retriever.invoke(claim)
        context = "\n".join(d.page_content for d in docs)
        label = llm.invoke(
            [SystemMessage(SUPPORT_SYSTEM), HumanMessage(f"Claim: {claim}\n\nContext:\n{context}")]
        )
        evidence.append(context)
        labels.append(label.content.strip().upper())
    print(f"  [grade] {labels}")
    return {"evidence": evidence, "support_labels": labels}


def needs_revision(state: SpeculativeState) -> str:
    return "revise" if any(l != "SUPPORTED" for l in state["support_labels"]) else "end"


def revise(state: SpeculativeState) -> dict:
    evidence_block = "\n\n".join(
        f"Claim: {c}\nStatus: {l}\nEvidence: {e}"
        for c, l, e in zip(state["claims"], state["support_labels"], state["evidence"])
    )
    result = llm.invoke(
        [
            SystemMessage(REVISE_SYSTEM),
            HumanMessage(f"Original:\n{state['draft']}\n\nFact-check:\n{evidence_block}"),
        ]
    )
    print("  [revise] answer updated")
    return {"revised": result.content}


graph = StateGraph(SpeculativeState)
graph.add_node("draft", draft)
graph.add_node("extract_claims", extract_claims)
graph.add_node("retrieve_and_grade", retrieve_and_grade)
graph.add_node("revise", revise)
graph.set_entry_point("draft")
graph.add_edge("draft", "extract_claims")
graph.add_edge("extract_claims", "retrieve_and_grade")
graph.add_conditional_edges("retrieve_and_grade", needs_revision, {"revise": "revise", "end": END})
graph.add_edge("revise", END)
app = graph.compile()
print("Graph compiled.")

In [ ]:
# ── 5. Run speculative RAG ────────────────────────────────────────────────────
QUERY = "When was LangGraph released and who made it?"

print(f"QUERY: {QUERY}\n")
result = app.invoke(
    {"query": QUERY, "draft": "", "claims": [], "evidence": [], "support_labels": [], "revised": ""}
)

print("\n=== DRAFT ===")
print(result["draft"])
print("\n=== CLAIMS & LABELS ===")
for claim, label in zip(result["claims"], result["support_labels"]):
    print(f"  [{label}] {claim}")
if result["revised"]:
    print("\n=== REVISED ANSWER ===")
    print(result["revised"])
else:
    print("\nAll claims supported — no revision needed.")

## Exercises

**Exercise 1 — Ask something not in the knowledge base:** Change `QUERY` to `"What is the capital of France?"`. Are claims SUPPORTED or UNRELATED? Does revision fire?

**Exercise 2 — Inject a wrong fact:** Add `"LangGraph was founded in 2019."` to `DOCS`. Ask the same query. Does the CONTRADICTED label appear and does the revision fix it?

**Exercise 3 — Count retrieval calls:** Add a counter inside `retrieve_and_grade`. For questions the model gets right, how many retrieval calls does speculative RAG save vs always-retrieve?

**Exercise 4 — Your own documents:** Replace `DOCS` with 10 sentences from any Wikipedia article. Ask 3 questions — one the model knows, one it doesn't, one where it might hallucinate.

## What's next

- **17-corrective-rag** — correct after retrieval (retrieve first, then grade)
- **27-self-rag** — decide per sentence whether to retrieve
- **25-adaptive-rag** — classify query upfront to pick cheapest retrieval strategy